### 12. PSD 센서 모니터링

#### 1) 모듈 추가

In [ ]:
import libraries.Omniwheel_Protocol as Omniwheel_Protocol
import serial
import time

#### 2) 프로토콜 정의

In [ ]:
Arduino_ID = 0x20
REQUEST_PSD_SENSOR = 0xA1
ANSWER_PSD_SENSOR = 0xB1
MID_PSD_SENSOR =0x81

#### 3) 변수 초기화

In [ ]:
data_PSD_Sensor = 0

#### 4) 송수신 객체 생성 및 초기화

In [ ]:
send_packet = Omniwheel_Protocol.Packet()
recv_packet = Omniwheel_Protocol.Packet()

In [ ]:
recv_packet.clearPacket()
send_packet.clearPacket()

#### 5) 변수 선언

In [ ]:
recv_list = []
recv_parsing_packet = []

In [ ]:
send_flag=False

#### 6) 통신 포트 정보 초기화

In [ ]:
Serial_Arduino = serial.Serial(port ="/dev/ttyACM0" ,baudrate = 115200, timeout=.1)
time.sleep(1)
print("connect complete")

#### 7) 송신 함수(센서 값 요청)

In [ ]:
def Packet_send(_id, _cmd, _mid, _data = None):
    # 전역변수 사용
    global send_flag
    
    # 전송 완료 플래그 확인 
    if(send_flag==False):
        
        # 초기화
        send_packet.clearPacket()
        
        # ID 설정
        send_packet.setID(_id)
        
        # CMD 설정
        send_packet.setCMD(_cmd)
        
        # Payload 초기화
        send_packet.clearPayload()
        
        # MID, data 설정
        send_packet.addPayload(_mid, _data)
        
        # 패킷 LRC 계산
        send_packet.calcLRC_Lower()
        
        # 패킷을 리스트로 변환
        send_list=send_packet.packetToList()
        
        # Arduino 에 패킷 리스트 전송
        Serial_Arduino.write(send_list)
        
        # 전송 완료 플래그 설정
        send_flag=True

#### 8) 수신 함수(데이터 수신)

In [ ]:
def Packet_receive(ser):
    # 전역변수 사용
    global send_flag
    
    # 전송 완료 플래그 확인
    if(send_flag==True):
        
        # 수신받은 데이터가 없을때까지
        while ser.inWaiting() > 0:
            
            # 1바이트씩 데이터를 받음
            Arduino_Data = ser.read(1)
            
            # 데이터를 받은 경우
            if(len(Arduino_Data)>0):
                
                # 수신 패킷 리스트에 수신 데이터 저장
                recv_list.append(ord(Arduino_Data))
                
                # 패킷 종료 데이터를 받은 경우
                if(ord(Arduino_Data)==0x03):
                    
                    # 수신 받은 데이터 파싱
                    if(recv_packet.parsingList(recv_list)):
                        
                        # 파싱한 데이터를 파싱완료 리스트에 저장
                        recv_parsing_packet.append(recv_packet)
                        
                        # 패킷 리스트 초기화
                        recv_list.clear()
                        
                        # 전송 완료 플래그 해제
                        send_flag=False
                        
                        # 반복문 탈출
                        break

#### 9) 수신 리스트 초기화 함수

In [ ]:
def Received_packet():
    # 파싱 완료 리스트의 첫번째 패킷을 result 변수에 저장함
    result=recv_parsing_packet[0]
    
    # 저장 완료한 파싱 완료 리스트 삭제 
    del recv_parsing_packet[0]
    
    # result 변수값 반환
    return result

#### 10) 센서 데이터 저장 함수

In [ ]:
# 인자값으로 Received_packet() 함수에서 반환된 값을 넣어줌
def PSD_Data(packet):
    
    #전역함수 사용
    global data_PSD_Sensor
    
    # 패킷에서 ID 추출
    packet_id=packet.getID()
    
    # 패킷에서 CMD 추출
    packet_cmd=packet.getCMD()
    
    # 추출한 ID 가 Arduino_ID 와 맞는지 확인
    if(packet_id==Arduino_ID):
        
        #추출한 CMD가 응답 CMD가 맞는지 확인
        if(packet_cmd==ANSWER_PSD_SENSOR):
            
            # 패킷에서 Payload 값을 추출함
            for payload in packet.getPayload():
                
                # 추출한 MID가 PSD 센서가 맞는지 확인
                if(payload.getID()==MID_PSD_SENSOR):
                    
                    # Payload 에서 읽어온 데이터를 버퍼에 저장 
                    buf=str(payload.getData())
                    
                    # 버퍼에 있는 데이터 값(PSD 센서 거리 값) 을 확인할수 있게 배열로 나누어 저장 
                    data_PSD_Sensor = [int(float(buf[:3])), int(float(buf[3:6])), int(float(buf[6:]))]

#### 11) 센서 값 모니터링

In [ ]:
while(True):
    # 데이터 요청 패킷 송신
    Packet_send(Arduino_ID, REQUEST_PSD_SENSOR, MID_PSD_SENSOR)
    
    # 응답 데이터 패킷 수신
    Packet_receive(Serial_Arduino)
    
    # 응답받은 데이터가 있을경우
    if(len(recv_parsing_packet) > 0):
        
        # 파싱완료 리스트에서 첫번째 패킷을 가져옴
        p=Received_packet()
        
        # 수신한 패킷을 파싱하고 PSD 센서 거리값을 저장
        PSD_Data(p)
        
        # 데이터 출력
        print("PSD : "+str(data_PSD_Sensor))
        
    # 일정시간 대기
    time.sleep(0.5)